In [ ]:
# ============================================
# CELL 0 — Install deps (run once per runtime)
# ============================================
!pip install -q pandas numpy pyarrow fastparquet spacy tqdm

In [ ]:
#!python -m spacy download en_core_web_sm -q

In [ ]:


# ============================================
# CELL 1 — Mount Drive & define paths
# ============================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, html, unicodedata, hashlib
import pandas as pd
import numpy as np

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"   # change if your folder differs

# INPUT (your screenshot folder)
INPUT_FILE_DRIVE = f"{BASE}/preprocess/reddit_common/reddit_corpus_2023_2025.csv"
INPUT_FILE_LOCAL = "/mnt/data/reddit_corpus_2023_2025.csv"   # fallback (uploaded here)
INPUT_FILE = INPUT_FILE_DRIVE if os.path.exists(INPUT_FILE_DRIVE) else INPUT_FILE_LOCAL

# Domain corpora (optional enrichers)
SINGLEX_CSV  = f"{BASE}/corpus/Singlish/lexicon.csv"
REGEX_JSONL  = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"
RULER_ORIG   = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
VOCAB_DIR    = f"{BASE}/corpus/SGPropertyDomain/vocab"            # *.txt category vocab
RULER_MERGED = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl"

# OUTPUTS (same place your labeling step expects)
OUTDIR       = f"{BASE}/preprocess/forum/reddit_common_2023_2025"
HASH_CHECKPT = f"{BASE}/labeled/checkpoints/reddit/reddit_common_hashes.parquet"
Path(OUTDIR).mkdir(parents=True, exist_ok=True)
Path(Path(HASH_CHECKPT).parent).mkdir(parents=True, exist_ok=True)

print("Using INPUT_FILE:", INPUT_FILE)
print("Saving to OUTDIR:", OUTDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using INPUT_FILE: /content/drive/MyDrive/PropInsight/preprocess/reddit_common/reddit_corpus_2023_2025.csv
Saving to OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_common_2023_2025


In [ ]:
# ============================================
# CELL 2 — Helpers (cleaning, regex, Singlish)
# ============================================
import glob
from tqdm.auto import tqdm

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def clean_basic(text: str) -> str:
    if not isinstance(text, str): return ""
    s = html.unescape(text)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\[.*?\]\(https?://[^\s)]+\)", " ", s)                # markdown [x](url)
    s = re.sub(r"https?://\S+|www\.\S+|\S+@\S+\.\S+", " ", s)        # urls/emails
    s = re.sub(r"^\s*\[(deleted|removed)\]\s*$", " ", s, flags=re.I)  # deleted markers
    s = re.sub(r"<[^>]+>", " ", s)                                    # html tags
    s = re.sub(r"([.,!?;:]){2,}", r"\1", s)
    return normalize_ws(s)

def compile_regexes_from_jsonl(path: Path):
    pats=[]
    if not path.exists(): return pats
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if not ln: continue
            try:
                d=json.loads(ln)
                pat=d.get("pattern"); name=d.get("name","pattern")
                if pat: pats.append((f"rx_{name}", re.compile(pat, re.I)))
            except: pass
    return pats

def build_singlish_dict(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column missing in {csv_path}. Found: {df.columns.tolist()}")
    words_set = set(df["word"].dropna().astype(str).str.strip().str.lower())
    return words_set

def find_singlish_terms(text: str, words_set):
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+", text or "")
    return sorted(set([t.lower() for t in toks if t.lower() in words_set]))

# ============================================
# CELL 3 — Merge vocab/*.txt into EntityRuler
# ============================================
def merge_vocab_to_entityruler(vocab_dir: str, existing_jsonl: str, merged_out: str):
    p_voc, p_ex, p_out = Path(vocab_dir), Path(existing_jsonl), Path(merged_out)
    existing=[]
    if p_ex.exists():
        for ln in p_ex.read_text(encoding="utf-8", errors="ignore").splitlines():
            ln=ln.strip()
            if ln:
                try: existing.append(json.loads(ln))
                except: pass

    def phrase_to_token_pattern(phrase: str):
        phrase = re.sub(r"\s+", " ", phrase).strip()
        if not phrase: return None
        return [{"LOWER": t.lower()} for t in phrase.split(" ") if t]

    def lowers_from_pattern(pat):
        if isinstance(pat, str): return tuple(re.sub(r"\s+"," ",pat).lower().split(" "))
        if isinstance(pat, dict): return (str(pat.get("LOWER", pat.get("TEXT",""))).lower(),)
        if isinstance(pat, list):
            outs=[]
            for tok in pat:
                if isinstance(tok, dict): outs.append(str(tok.get("LOWER", tok.get("TEXT",""))).lower())
                else: outs.append(str(tok).lower())
            return tuple(outs)
        return (str(pat).lower(),)

    def pat_key(rec): return (rec.get("label",""), lowers_from_pattern(rec.get("pattern","")))

    merged=[]; seen=set()
    for rec in existing:
        k = pat_key(rec)
        if k not in seen: seen.add(k); merged.append(rec)

    if p_voc.exists():
        for txt in sorted(p_voc.glob("*.txt")):
            label = re.sub(r"[^A-Za-z0-9]+","_", txt.stem).strip("_").upper() or "DOMAIN"
            for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
                term = raw.strip()
                if not term: continue
                pat = phrase_to_token_pattern(term)
                if not pat: continue
                rec={"label":label,"pattern":pat,"id":term}
                k=pat_key(rec)
                if k not in seen: seen.add(k); merged.append(rec)

    p_out.parent.mkdir(parents=True, exist_ok=True)
    with p_out.open("w", encoding="utf-8") as f:
        for rec in merged:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[EntityRuler] merged → {p_out} (rules: {len(merged)})")
    return str(p_out)

ENTITYRULER_MERGED = merge_vocab_to_entityruler(VOCAB_DIR, RULER_ORIG, RULER_MERGED)

# ============================================
# CELL 4 — Load CSV and normalize columns
# ============================================
df = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)
# Try to be robust to different source schemas
for c in ["title","selftext","comment_body","url","permalink","subreddit","author","comment_author"]:
    if c not in df.columns: df[c] = ""

# unify timestamps → prefer comment timestamp if present
ts_comment = None
for cand in ["created_utc_comment","created_utc_c","created_comment_utc","comment_time","comment_created_utc"]:
    if cand in df.columns: ts_comment=cand; break
ts_post = None
for cand in ["created_utc_post","created_utc","post_time","post_created_utc"]:
    if cand in df.columns: ts_post=cand; break

if (ts_comment is None) and (ts_post is None):
    # best effort: if 'date' exists, keep; else create NA
    if "date" not in df.columns: df["date"] = pd.NaT

# ============================================
# CELL 5 — Build date, clean body, window filter
# ============================================
if "date" not in df.columns:
    dt = pd.to_datetime(df.get(ts_comment, ""), errors="coerce", utc=True)
    dt = dt.fillna(pd.to_datetime(df.get(ts_post, ""), errors="coerce", utc=True))
    df["date"] = dt

# Ensure 'date' column is datetime type before timezone conversion
if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

# Filter out NaT values before timezone conversion
df = df.dropna(subset=["date"]).copy()

df["date"] = df["date"].dt.tz_convert("Asia/Singapore")
mask_window = (df["date"] >= "2023-01-01") & (df["date"] < "2026-01-01")
before = len(df); df = df[mask_window].copy()
print(f"[Date Filter] Kept {len(df)} / {before} rows (2023–2025)")

df["title"]        = df["title"].astype(str)
df["selftext"]     = df["selftext"].astype(str)
df["comment_body"] = df["comment_body"].astype(str)

if "body" not in df.columns:
    df["body_raw"] = (df["title"].fillna("") + "\n\n" +
                      df["selftext"].fillna("") + "\n\n" +
                      df["comment_body"].fillna("")).str.strip()
    df["body"] = df["body_raw"].apply(clean_basic)
else:
    df["body"] = df["body"].astype(str).apply(clean_basic)

# Low-signal filters
n0 = len(df); df = df[df["body"].str.len() >= 60].copy()
print(f"[Low-signal] Removed {n0-len(df)} short rows")
n0 = len(df); df = df[df["body"].str.contains(r"[A-Za-z]", na=False)].copy()
print(f"[Alpha-check] Removed {n0-len(df)} rows with no alphabetic characters")

# Temporal partitions
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.to_period("M").astype(str)
df["quarter"] = df["date"].dt.to_period("Q").astype(str)

# ============================================
# CELL 6 — Dedupe (early key + body hash) & checkpoint
# ============================================
df["_k1"] = (
    df.get("url","").astype(str).str.lower() + "||" +
    df["title"].astype(str).str.lower() + "||" +
    df["selftext"].astype(str).str.lower().str[:300]
)
before = len(df); df = df.drop_duplicates("_k1").drop(columns=["_k1"])
print(f"[Dedupe-early] Removed {before-len(df)} rows")

df["hash"] = df["body"].str.lower().map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())
before = len(df); df = df.drop_duplicates("hash").copy()
print(f"[Dedupe-hash] Removed {before-len(df)} rows; kept {len(df)}")

try:
    seen = pd.read_parquet(HASH_CHECKPT)
    seen_set = set(seen["hash"].astype(str))
except Exception:
    seen_set = set()
print("New unique bodies (vs global store):", (~df["hash"].isin(seen_set)).sum())

# ============================================
# CELL 7 — Enrichment: regex + Singlish + EntityRuler
# ============================================
# 7a) Regex flags
pats = compile_regexes_from_jsonl(Path(REGEX_JSONL))
if pats:
    for col, rx in pats:
        try: df[col] = df["body"].str.contains(rx, na=False)
        except Exception: df[col] = False
    print(f"[Regex] Added {len(pats)} rx_* columns")
else:
    print("[Regex] No patterns found; skipping")

# 7b) Singlish
try:
    words = build_singlish_dict(SINGLEX_CSV)
    df["singlish_terms"] = df["body"].apply(lambda s: find_singlish_terms(s, words))
    df["has_singlish"]   = df["singlish_terms"].str.len().gt(0)
    print(f"[Singlish] Terms detected in {df['has_singlish'].sum()} rows")
except Exception as e:
    print("[Singlish] Skipped:", e)
    df["singlish_terms"] = [[] for _ in range(len(df))]
    df["has_singlish"]   = False

# 7c) EntityRuler (domain entities only; fast)
try:
    import spacy
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler")
    ruler.from_disk(RULER_MERGED)
    ents=[]
    for doc in nlp.pipe(df["body"].astype(str).tolist(), batch_size=64):
        ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
    df["entities"] = ents
    print(f"[EntityRuler] Entities attached: {len(df)} rows")
except Exception as e:
    print("[EntityRuler] Skipped:", e)
    df["entities"] = [[] for _ in range(len(df))]

# ============================================
# CELL 8 — Save main clean corpus (CSV/Parquet)
# ============================================
core_cols = [
    "hash","date","year","quarter","month",
    "subreddit","post_id","comment_id","author","comment_author",
    "title","selftext","comment_body","body",
    "url","permalink","score_post","comment_score","num_comments"
]
rx_cols    = [c for c in df.columns if c.startswith("rx_")]
extra_cols = ["has_singlish","singlish_terms","entities"]

df_out = df[[c for c in core_cols if c in df.columns] + rx_cols + extra_cols].copy()

csv_path  = f"{OUTDIR}/reddit_clean_common_2023_2025.csv"
parq_path = f"{OUTDIR}/reddit_clean_common_2023_2025.parquet"
df_out.to_csv(csv_path, index=False)
try: df_out.to_parquet(parq_path, index=False)
except Exception as e: print("[WARN] Parquet save failed:", e)

print("Saved CSV:", csv_path)
print("Saved Parquet:", parq_path)
print("Rows saved:", len(df_out))

[EntityRuler] merged → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl (rules: 2158)
[Date Filter] Kept 61918 / 61918 rows (2023–2025)
[Low-signal] Removed 166 short rows
[Alpha-check] Removed 0 rows with no alphabetic characters


/tmp/ipython-input-214005515.py:164: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"]   = df["date"].dt.to_period("M").astype(str)
/tmp/ipython-input-214005515.py:165: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["quarter"] = df["date"].dt.to_period("Q").astype(str)


[Dedupe-early] Removed 58572 rows
[Dedupe-hash] Removed 0 rows; kept 3180
New unique bodies (vs global store): 3180
[Regex] No patterns found; skipping
[Singlish] Terms detected in 2881 rows
[EntityRuler] Entities attached: 3180 rows
Saved CSV: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_common_2023_2025/reddit_clean_common_2023_2025.csv
Saved Parquet: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_common_2023_2025/reddit_clean_common_2023_2025.parquet
Rows saved: 3180


In [ ]:
# ============================================
# CELL 9 — OPTIONAL heavy NLP (spaCy model)
# ============================================
RUN_NLP_ENRICHMENT = True   # flip True if you want tokens/lemmas/ner/sentences

if RUN_NLP_ENRICHMENT:
    import spacy, sys, subprocess
    try:
        nlp2 = spacy.load("en_core_web_sm", exclude=[])
    except:
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
        nlp2 = spacy.load("en_core_web_sm", exclude=[])

    try:
        er = nlp2.add_pipe("entity_ruler", before="ner")
        er.from_disk(RULER_MERGED)
    except Exception as e:
        print("[WARN] Could not attach EntityRuler:", e)

    texts = df_out["body"].astype(str).tolist()
    docs = list(nlp2.pipe(texts, batch_size=64, n_process=2))

    df_out["tokens"]         = [[t.text for t in d] for d in docs]
    df_out["lemmas"]         = [[t.lemma_ for t in d] for d in docs]
    df_out["pos"]            = [[t.pos_ for t in d] for d in docs]
    df_out["deps"]           = [[t.dep_ for t in d] for d in docs]
    df_out["entities_ner"]   = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
    df_out["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]

    enr_dir = Path(OUTDIR) / "nlp_enriched"; enr_dir.mkdir(parents=True, exist_ok=True)
    df_out.to_parquet(enr_dir / "reddit_common_enriched+nlp.parquet", index=False)
    print("Saved:", enr_dir / "reddit_common_enriched+nlp.parquet")

    # Sentence table
    sent_rows=[]
    for i, d in enumerate(docs):
        for j, s in enumerate(d.sents):
            sent_rows.append({
                "doc_id": i, "sent_id": j, "text": s.text,
                "tokens": [t.text for t in s],
                "lemmas": [t.lemma_ for t in s],
                "pos":    [t.pos_ for t in s],
                "deps":   [t.dep_ for t in s],
                "date":   df_out.iloc[i]["date"]
            })
    pd.DataFrame(sent_rows).to_parquet(enr_dir / "reddit_common_sentences.parquet", index=False)
    print("Saved:", enr_dir / "reddit_common_sentences.parquet")
else:
    print("[INFO] RUN_NLP_ENRICHMENT=False → skip heavy spaCy step.")

Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_common_2023_2025/nlp_enriched/reddit_common_enriched+nlp.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_common_2023_2025/nlp_enriched/reddit_common_sentences.parquet


In [ ]:
# ============================================
# CELL 10 — Update global hash checkpoint
# ============================================
try:
    cur = pd.read_parquet(HASH_CHECKPT)
except Exception:
    cur = pd.DataFrame(columns=["hash"])
to_add = df_out[["hash"]].drop_duplicates()
merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
merged.to_parquet(HASH_CHECKPT, index=False)
print("Updated hash checkpoint:", HASH_CHECKPT, "| total hashes:", len(merged))

Updated hash checkpoint: /content/drive/MyDrive/PropInsight/labeled/checkpoints/reddit/reddit_common_hashes.parquet | total hashes: 3180
